# Pilot Data Analysis (2026-03-02)

Analysis of pilot data from the face-flanker memory experiment. 8 subjects, all in condition 1 (item recognition). This is the first data collection after the March 2 design changes: face cropping (`FACE_CROP=0.5`), removed on-screen trial prompts, practice-only prompts with key quiz, rest break key reminders, fixed M=male study key mapping, and halved study list (60 trials). See `../notes/data-guide.md` for the data dictionary.

### Design recap

**Study phase.** On each trial, a neutral target face appears flanked on both sides by the same emotional face (angry, happy, or neutral). The participant judges the target's gender (Z=female, M=male). The display stays on screen for a fixed 3 seconds regardless of response. The factorial design crosses target gender (2) × flanker gender (2) × flanker emotion (3) = 12 trial types, each repeated 5 times, for 60 study trials per subject.

**Test phase (condition 1 — item recognition).** Each participant sees 120 single neutral faces (60 old + 60 new) and judges each as old or new. The old/new key mapping is counterbalanced across subjects via `key_mapping`.

In [8]:
import pandas as pd
from pathlib import Path
from statistics import NormalDist

z = NormalDist().inv_cdf  # equivalent to scipy.stats.norm.ppf

# Resolve CSV path relative to this notebook's location
_nb_dir = Path.cwd() if '__vsc_ipynb_file__' not in dir() else Path(__vsc_ipynb_file__).parent
df = pd.read_csv(_nb_dir / '2026_03_02_pilot_8subj.csv')
df['correct'] = df['correct'].astype('boolean')  # nullable bool

print(f'{len(df)} rows, {df.subject_number.nunique()} subjects')
print(f'Conditions: {df.groupby("condition").subject_number.nunique().to_dict()}')

1440 rows, 8 subjects
Conditions: {1: 8}


## Data Overview

In [9]:
overview = df.groupby(['subject_number', 'condition']).apply(
    lambda g: pd.Series({
        'study_trials': (g.phase == 'study').sum(),
        'test_trials': (g.phase == 'test').sum(),
        'total': len(g),
        'timeouts': g.timed_out.sum()
    })
).reset_index()
print(f'{len(overview)} subjects')
print()
print('Trial counts:')
n = len(overview)
study_min, study_max = overview.study_trials.min(), overview.study_trials.max()
test_min, test_max = overview.test_trials.min(), overview.test_trials.max()
timeout_med = overview.timeouts.median()
timeout_max = overview.timeouts.max()
study_str = str(int(study_min)) if study_min == study_max else f'{int(study_min)}-{int(study_max)}'
test_str = str(int(test_min)) if test_min == test_max else f'{int(test_min)}-{int(test_max)}'
print(f'  Condition 1 (n={n}): study={study_str}, test={test_str}, '
      f'timeouts median={timeout_med:.0f} max={timeout_max:.0f}')
print()
print('Subjects with >5 timeouts:')
high_timeout = overview[overview.timeouts > 5]
if len(high_timeout) == 0:
    print('  None')
else:
    print(high_timeout[['subject_number', 'condition', 'timeouts']].to_string(index=False))

8 subjects

Trial counts:
  Condition 1 (n=8): study=60, test=120, timeouts median=0 max=4

Subjects with >5 timeouts:
  None


## Study Phase Analysis

The study phase uses the same 2 × 2 × 3 factorial (target gender × flanker gender × flanker emotion) but with 5 reps per cell (60 trials total, down from 120 in the Feb 20 pilot). Study key mapping is now fixed: Z=female, M=male.

Analyzing study-phase performance checks two things:

1. **Cover task validation** — accuracy should be near ceiling, confirming participants are attending to the faces.
2. **Flanker interference** — RT differences across flanker emotions would indicate the flankers are being processed. Because the display is shown for a fixed 3 seconds regardless of response time, RT differences do not create an encoding-time confound.

In [10]:
study = df[df.phase == 'study'].copy()

# Per-subject accuracy
subj_acc = study.groupby('subject_number').correct.mean()
print(f'Study phase: {len(study)} trials from {study.subject_number.nunique()} subjects')
print(f'Overall accuracy: {study.correct.mean():.1%}')
print(f'Per-subject accuracy: M={subj_acc.mean():.3f}, SD={subj_acc.std():.3f}, '
      f'range=[{subj_acc.min():.3f}, {subj_acc.max():.3f}]')
print()

# Accuracy and RT by flanker emotion
study_by_emotion = study.groupby('flanker_emotion').apply(
    lambda g: pd.Series({
        'N': len(g),
        'accuracy': g.correct.mean(),
        'mean_rt': g.loc[~g.timed_out, 'rt'].mean(),
        'sd_rt': g.loc[~g.timed_out, 'rt'].std(),
        'timeouts': g.timed_out.sum()
    })
).round(3)
print('Accuracy and RT by flanker emotion (pooled across subjects):')
print(study_by_emotion.to_string())

Study phase: 480 trials from 8 subjects
Overall accuracy: 72.5%
Per-subject accuracy: M=0.725, SD=0.351, range=[0.050, 1.000]

Accuracy and RT by flanker emotion (pooled across subjects):
                     N  accuracy  mean_rt    sd_rt  timeouts
flanker_emotion                                             
angry            160.0     0.738  951.138  458.740       0.0
happy            160.0     0.719  913.962  409.379       1.0
neutral          160.0     0.719  922.572  383.291       1.0


## RQ1: Does Emotional Context Affect Recognition Accuracy? (Condition 1)

At test, participants see single neutral faces and judge each as old (studied) or new. The test includes 60 studied targets plus 60 novel faces (120 total, down from 240 in the Feb 20 pilot). Each studied target was originally flanked by angry, happy, or neutral faces (20 targets per emotion), but the flanker is not shown at test.

**Signal detection.** d' separates genuine memory (sensitivity) from response tendencies (criterion c). The false alarm rate is pooled across all new items because new items have no flanker emotion assignment, so d' differences across emotions are driven entirely by hit rate differences.

**Reading d':** 0 = chance, ~0.5 = weak, ~1.0 = moderate, ~2.0 = strong.  
**Reading criterion c:** 0 = unbiased, positive = conservative (tends to say "new"), negative = liberal (tends to say "old").

In [11]:
def edge_correct(rate, n):
    """Apply Macmillan & Kaplan (1985) edge correction."""
    if rate == 0:
        return 0.5 / n
    if rate == 1:
        return 1 - 0.5 / n
    return rate


c1_test = df[(df.condition == 1) & (df.phase == 'test')].copy()
n_subj_c1 = c1_test.subject_number.nunique()

# Per-subject SDT metrics
rq1_per_subj = []
for subj, sdata in c1_test.groupby('subject_number'):
    new_items = sdata[sdata.stimulus_type == 'new']
    fa_rate_raw = (~new_items.correct).mean()
    fa_n = len(new_items)
    fa_rate = edge_correct(fa_rate_raw, fa_n)

    old_items = sdata[sdata.stimulus_type == 'old']
    for emotion, grp in old_items.groupby('study_flanker_emotion'):
        n = len(grp)
        hit_rate_raw = grp.correct.mean()
        hit_rate = edge_correct(hit_rate_raw, n)
        dprime = z(hit_rate) - z(fa_rate)
        criterion = -0.5 * (z(hit_rate) + z(fa_rate))
        rq1_per_subj.append({
            'subject': subj,
            'emotion': emotion,
            'hit_rate': hit_rate_raw,
            'fa_rate': fa_rate_raw,
            'd_prime': dprime,
            'criterion': criterion
        })

rq1_df = pd.DataFrame(rq1_per_subj)

# Group summary
rq1_summary = rq1_df.groupby('emotion').agg(
    N=('subject', 'count'),
    hit_rate_M=('hit_rate', 'mean'),
    hit_rate_SD=('hit_rate', 'std'),
    fa_rate_M=('fa_rate', 'mean'),
    d_prime_M=('d_prime', 'mean'),
    d_prime_SD=('d_prime', 'std'),
    criterion_M=('criterion', 'mean'),
    criterion_SD=('criterion', 'std'),
).round(3)

print(f'RQ1: Item Recognition (n={n_subj_c1} subjects)')
print(f'Mean FA rate: {rq1_df.fa_rate.mean():.3f} (SD={rq1_df.groupby("subject").fa_rate.first().std():.3f})')
print()
print(rq1_summary.to_string())

RQ1: Item Recognition (n=8 subjects)
Mean FA rate: 0.310 (SD=0.188)

         N  hit_rate_M  hit_rate_SD  fa_rate_M  d_prime_M  d_prime_SD  criterion_M  criterion_SD
emotion                                                                                         
angry    8       0.525        0.187       0.31      0.640       0.604        0.253         0.470
happy    8       0.456        0.152       0.31      0.451       0.454        0.348         0.469
neutral  8       0.406        0.143       0.31      0.314       0.585        0.416         0.428


In [12]:
# Per-subject d' by emotion (for inspecting individual variation)
rq1_pivot = rq1_df.pivot(index='subject', columns='emotion', values='d_prime').round(3)
print("Per-subject d' by flanker emotion:")
print(rq1_pivot.to_string())

Per-subject d' by flanker emotion:
emotion  angry  happy  neutral
subject                       
1        0.876  0.623    0.497
2        1.152  0.730    0.603
3        0.512  0.195    0.000
4       -0.334  0.215   -0.184
5        1.519  1.392    1.392
6        0.981  0.204    0.343
7        0.244 -0.045   -0.556
8        0.168  0.294    0.421


### Statistical Tests

One-way repeated-measures ANOVA on d' (factor: flanker emotion, 3 levels) tests whether recognition sensitivity differs across emotional contexts. Paired t-tests follow up on specific comparisons. All tests are within-subjects (n=8).

In [13]:
import math

# --- Exact t and F p-values via incomplete beta function (no scipy needed) ---

def _betacf(a, b, x):
    """Continued fraction for regularized incomplete beta (Numerical Recipes)."""
    MAXIT, EPS = 200, 3e-12
    qab, qap, qam = a + b, a + 1.0, a - 1.0
    c = 1.0
    d = max(abs(1.0 - qab * x / qap), 1e-30)
    d = 1.0 / d if (1.0 - qab * x / qap) >= 0 else -1.0 / d
    # Fix sign
    d = 1.0 / (1.0 - qab * x / qap) if abs(1.0 - qab * x / qap) > 1e-30 else 1.0 / 1e-30
    h = d
    for m in range(1, MAXIT + 1):
        m2 = 2 * m
        aa = m * (b - m) * x / ((qam + m2) * (a + m2))
        d = 1.0 + aa * d
        if abs(d) < 1e-30: d = 1e-30
        c = 1.0 + aa / c
        if abs(c) < 1e-30: c = 1e-30
        d = 1.0 / d
        h *= d * c
        aa = -(a + m) * (qab + m) * x / ((a + m2) * (qap + m2))
        d = 1.0 + aa * d
        if abs(d) < 1e-30: d = 1e-30
        c = 1.0 + aa / c
        if abs(c) < 1e-30: c = 1e-30
        d = 1.0 / d
        delta = d * c
        h *= delta
        if abs(delta - 1.0) < EPS:
            break
    return h

def _betai(a, b, x):
    """Regularized incomplete beta function I_x(a, b)."""
    if x <= 0: return 0.0
    if x >= 1: return 1.0
    lbeta = math.lgamma(a) + math.lgamma(b) - math.lgamma(a + b)
    front = math.exp(a * math.log(x) + b * math.log(1 - x) - lbeta)
    if x < (a + 1) / (a + b + 2):
        return front * _betacf(a, b, x) / a
    else:
        return 1.0 - front * _betacf(b, a, 1 - x) / b

def t_p_twotail(t_val, df):
    """Exact two-tailed p-value for t-distribution."""
    x = df / (df + t_val ** 2)
    return _betai(df / 2.0, 0.5, x)

def f_p(f_val, df1, df2):
    """Upper-tail p-value for F-distribution."""
    if f_val <= 0: return 1.0
    x = df2 / (df2 + df1 * f_val)
    return _betai(df2 / 2.0, df1 / 2.0, x)

# --- Tests ---

def paired_ttest(x, y):
    """Paired t-test (two-tailed). Returns t, df, p, mean_diff, cohen_d_z."""
    diffs = [a - b for a, b in zip(x, y)]
    n = len(diffs)
    mean_d = sum(diffs) / n
    var_d = sum((d - mean_d) ** 2 for d in diffs) / (n - 1)
    se_d = math.sqrt(var_d / n)
    if se_d == 0:
        return 0.0, n - 1, 1.0, mean_d, 0.0
    t_val = mean_d / se_d
    p = t_p_twotail(t_val, n - 1)
    d_z = mean_d / math.sqrt(var_d)
    return t_val, n - 1, p, mean_d, d_z

def rm_anova_oneway(groups):
    """One-way repeated-measures ANOVA. Returns F, df1, df2, p, partial_eta_sq."""
    k = len(groups)
    n = len(groups[0])
    grand = sum(sum(g) for g in groups) / (k * n)
    subj_m = [sum(groups[j][i] for j in range(k)) / k for i in range(n)]
    cond_m = [sum(g) / n for g in groups]
    ss_cond = n * sum((m - grand) ** 2 for m in cond_m)
    ss_subj = k * sum((m - grand) ** 2 for m in subj_m)
    ss_total = sum((groups[j][i] - grand) ** 2 for j in range(k) for i in range(n))
    ss_err = ss_total - ss_cond - ss_subj
    df1 = k - 1
    df2 = (k - 1) * (n - 1)
    f_val = (ss_cond / df1) / (ss_err / df2) if ss_err > 0 else float('nan')
    p = f_p(f_val, df1, df2)
    eta = ss_cond / (ss_cond + ss_err)
    return f_val, df1, df2, p, eta

# Pivot d' to wide format
dprime_wide = rq1_df.pivot(index='subject', columns='emotion', values='d_prime')
angry = dprime_wide['angry'].tolist()
happy = dprime_wide['happy'].tolist()
neutral = dprime_wide['neutral'].tolist()

# Repeated-measures ANOVA
f_val, df1, df2, p_anova, eta_sq = rm_anova_oneway([angry, happy, neutral])
print(f"One-way RM ANOVA on d' (flanker emotion):")
print(f"  F({df1},{df2}) = {f_val:.3f}, p = {p_anova:.3f}, partial eta^2 = {eta_sq:.3f}")
print()

# Pairwise paired t-tests
pairs = [('angry', 'neutral', angry, neutral),
         ('angry', 'happy', angry, happy),
         ('happy', 'neutral', happy, neutral)]

print("Pairwise paired t-tests:")
for label_a, label_b, vals_a, vals_b in pairs:
    t, dof, p, md, dz = paired_ttest(vals_a, vals_b)
    print(f"  {label_a} vs {label_b}: t({dof}) = {t:.3f}, p = {p:.3f}, "
          f"mean diff = {md:.3f}, Cohen's d_z = {dz:.3f}")

One-way RM ANOVA on d' (flanker emotion):
  F(2,14) = 3.637, p = 0.053, partial eta^2 = 0.342

Pairwise paired t-tests:
  angry vs neutral: t(7) = 2.421, p = 0.046, mean diff = 0.325, Cohen's d_z = 0.856
  angry vs happy: t(7) = 1.362, p = 0.215, mean diff = 0.189, Cohen's d_z = 0.481
  happy vs neutral: t(7) = 1.662, p = 0.140, mean diff = 0.136, Cohen's d_z = 0.588


### RQ1 Interpretation

**Recognition performance improved.** Mean d' across emotion conditions is 0.47 — roughly 50% higher than the Feb 20 pilot (d'~0.31 with n=31). Hit rates are higher (.41–.53 vs .41–.43) while the FA rate is comparable (.31 vs .32). The halved study list (60 vs 120) may be helping — fewer items to encode means stronger traces per item. Five of eight subjects show clearly above-chance recognition (d'>0.4 for at least two emotions).

**Emotion gradient emerges, marginally significant.** Unlike the flat pattern in the Feb 20 pilot, there is now a graded ordering: angry d'=0.64 > happy d'=0.45 > neutral d'=0.31. The omnibus RM ANOVA is marginal, F(2,14) = 3.64, p = .053, partial eta-sq = .34 — a large effect size that just misses significance at n=8. The critical comparison (angry vs neutral) is significant: t(7) = 2.42, p = .046, d_z = 0.86. Angry vs happy (p = .22) and happy vs neutral (p = .14) are not individually significant, consistent with a graded pattern rather than a categorical angry-vs-everything effect.

**Study phase accuracy is concerning.** Overall study accuracy dropped to 72.5%, well below the 96.5% ceiling from the Feb 20 pilot. One subject (subject 4) is at 5% — essentially pressing the wrong key on every trial. The fixed M=male mapping should make the task easier, not harder. Possible explanations:
- **Key confusion**: Despite the key quiz and practice, some subjects may have mapped keys incorrectly. Subject 4's 5% accuracy is consistent with a reversed mapping (pressing Z for male, M for female).
- **Reduced prompts**: Removing on-screen key reminders during real trials may be hurting more than expected, even with the quiz and rest break reminders.
- **Sample**: With only 8 subjects, one confused participant heavily affects the mean.

**Subject 4 likely has reversed keys.** 5% accuracy on a binary task means they were systematically wrong — almost certainly pressing the wrong key. Their test-phase d' is near or below zero across all emotions, consistent with general disengagement or confusion. This subject should probably be excluded from analysis.

**Timeouts much improved.** Maximum 4 timeouts per subject (vs 21 in the Feb 20 pilot). No subjects exceed the >5 threshold. The shorter study list and design changes seem to be keeping participants engaged.

**Bottom line.** The halved study list appears to have both improved overall recognition and revealed an emotion gradient (angry > happy > neutral) that was absent in the Feb 20 pilot. The effect size is large (partial eta-sq = .34) but the sample is too small for definitive inference. The study accuracy drop needs investigation — the key quiz and practice may not be sufficient scaffolding for some participants.

## Summary

In [14]:
print('=== Pilot Data Summary (2026-03-02) ===')
print(f'{len(df)} trials, {df.subject_number.nunique()} subjects (all condition 1)')
print()
print('Study phase:')
print(f'  Overall accuracy: {study.correct.mean():.1%}')
print(f'  Mean RT: {study.loc[~study.timed_out, "rt"].mean():.0f} ms')
print()
print(f'RQ1 - Item Recognition (n={n_subj_c1}):')
for emotion, row in rq1_summary.iterrows():
    print(f"  {emotion:>7}: d'={row['d_prime_M']:.2f} (SD={row['d_prime_SD']:.2f}), "
          f"c={row['criterion_M']:.2f}, hit={row['hit_rate_M']:.2f}")

=== Pilot Data Summary (2026-03-02) ===
1440 trials, 8 subjects (all condition 1)

Study phase:
  Overall accuracy: 72.5%
  Mean RT: 929 ms

RQ1 - Item Recognition (n=8):
    angry: d'=0.64 (SD=0.60), c=0.25, hit=0.53
    happy: d'=0.45 (SD=0.45), c=0.35, hit=0.46
  neutral: d'=0.31 (SD=0.58), c=0.42, hit=0.41
